# EcoFood — Google Trends

Notebook con tu código original, separado en celdas. Se mantienen los términos, Argentina, el período de 5 años, las pausas y los cálculos.

In [1]:
import time
import os
import pandas as pd
from pytrends.request import TrendReq

pytrends = TrendReq(hl="es-AR", tz=180)

terminos = ["alimentación saludable", "veganismo", "sin TACC",
            "café de especialidad", "alimentos naturales"]

## 1. Consulta conjunta

In [2]:
pytrends.build_payload(terminos, geo="AR", timeframe="today 5-y")
juntos = pytrends.interest_over_time()
juntos = juntos[juntos["isPartial"] == False].drop(columns="isPartial")
time.sleep(30)

display(juntos.head())

,alimentación saludable,veganismo,sin TACC,café de especialidad,alimentos naturales
date,,,,,
2021-11-07,9,5,50,0,3
2021-11-14,9,3,50,0,2
2021-11-21,8,3,50,0,3
2021-11-28,5,2,46,0,2
2021-12-05,4,2,49,0,1


## 2. Consultas individuales

In [3]:
series = {}
for t in terminos:
    pytrends.build_payload([t], geo="AR", timeframe="today 5-y")
    d = pytrends.interest_over_time()
    if t in d:
        series[t] = d[t][d["isPartial"] == False]
    print("listo:", t)
    time.sleep(30)

indiv = pd.DataFrame(series)

display(indiv.head())

listo: alimentación saludable
listo: veganismo
listo: sin TACC
listo: café de especialidad
listo: alimentos naturales


,alimentación saludable,veganismo,sin TACC,café de especialidad,alimentos naturales
date,,,,,
2021-11-07,89,60,50,0,81
2021-11-14,90,31,50,0,61
2021-11-21,83,34,50,0,73
2021-11-28,53,27,46,0,55
2021-12-05,35,23,49,0,32


## 3. Guardado de CSV

En Jupyter no existe `__file__`, así que se usa la carpeta actual para guardar los archivos.

In [ ]:
import os

carpeta = os.path.dirname(os.getcwd())

ruta = os.path.join(carpeta, "data", "raw", "trends_juntos.csv")

juntos.to_csv(ruta, index=False)

print("CSV guardado en:", ruta)

## 4. Información y resumen

In [ ]:
print("\nInterés de búsqueda relativo (escala común, 2022-2025)")
comun = juntos.loc["2022":"2025"]
print(comun.mean().sort_values(ascending=False).round(1))

print("\nPorcentaje de semanas en zona de ruido (índice <= 2)")
print(((comun <= 2).mean() * 100).round(0))

print("\nDirección de cada término (escala propia, promedio por año)")
print(indiv.loc["2022":"2025"].groupby(indiv.loc["2022":"2025"].index.year).mean().round(0))